## Project Configuration

In [0]:
catalog = "ecommerce"

landing_schema = "landing"
bronze_schema = "bronze"

volume = "ecommerce_data"

base_path = f"/Volumes/{catalog}/{landing_schema}/{volume}"

datasets = [
    "customers",
    "products",
    "orders",
    "payments",
    "returns"
]

## Generic Auto Loader Function

In [0]:
def ingest_to_bronze(dataset):

    input_path = f"{base_path}/{dataset}"

    schema_path     = f"/Volumes/ecommerce/landing/autoloader_meta/schemas/{dataset}"
    checkpoint_path = f"/Volumes/ecommerce/landing/autoloader_meta/checkpoints/{dataset}"

    table_name = f"{catalog}.{bronze_schema}.{dataset}"

    print(f"\nStarting ingestion for {dataset}")

    df = (
        spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "json")
            .option("cloudFiles.schemaLocation", schema_path)
            .load(input_path)
    )

    query = (
        df.writeStream
            .format("delta")
            .option("checkpointLocation", checkpoint_path)
            .option("mergeSchema", "true")      # Handles schema evolution
            .trigger(availableNow=True)         # Process available files and stop
            .toTable(table_name)
    )

    query.awaitTermination()

    print(f"{dataset} Bronze table created successfully.")

## Run Auto Loader for Every Dataset

In [0]:
for dataset in datasets:
    ingest_to_bronze(dataset)


Starting ingestion for customers
customers Bronze table created successfully.

Starting ingestion for products
products Bronze table created successfully.

Starting ingestion for orders
orders Bronze table created successfully.

Starting ingestion for payments
payments Bronze table created successfully.

Starting ingestion for returns
returns Bronze table created successfully.


## Verify Tables Exist

In [0]:
spark.sql(f"SHOW TABLES IN {catalog}.{bronze_schema}").show(truncate=False)

+--------+---------+-----------+
|database|tableName|isTemporary|
+--------+---------+-----------+
|bronze  |customers|false      |
|bronze  |orders   |false      |
|bronze  |payments |false      |
|bronze  |products |false      |
|bronze  |returns  |false      |
+--------+---------+-----------+



## Verify Record Counts

In [0]:
for dataset in datasets:

    table = f"{catalog}.{bronze_schema}.{dataset}"

    count = spark.table(table).count()

    print(f"{dataset:<12} -> {count} records")

customers    -> 1000 records
products     -> 5000 records
orders       -> 100000 records
payments     -> 100000 records
returns      -> 10000 records


## Check Schemas

In [0]:
for dataset in datasets:

    print(f"\n========== {dataset.upper()} ==========\n")

    spark.table(f"{catalog}.{bronze_schema}.{dataset}").printSchema()


========== CUSTOMERS ==========

root
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- date_of_birth: string (nullable = true)
 |-- email: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- loyalty_points: string (nullable = true)
 |-- membership: string (nullable = true)
 |-- operation: string (nullable = true)
 |-- operation_timestamp: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- postal_code: string (nullable = true)
 |-- registration_date: string (nullable = true)
 |-- state: string (nullable = true)
 |-- status: string (nullable = true)
 |-- _rescued_data: string (nullable = true)


========== PRODUCTS ==========

root
 |-- brand: string (nullable = true)
 |-- category: string (nullable = true)
 |-- cost_price: string (nullable = true)
 |-- launch_date: string (nullable = true)
 |-- opera

## Preview Data

In [0]:
display(spark.table("ecommerce.bronze.customers").limit(20))

city,country,customer_id,date_of_birth,email,first_name,gender,last_name,loyalty_points,membership,operation,operation_timestamp,phone,postal_code,registration_date,state,status,_rescued_data
Faisalabad,Pakistan,1,1970-02-09,mark.johnson1@email.com,Mark,Male,Johnson,4467,Bronze,INSERT,2025-06-25T18:49:24Z,+923074913853,23434,2024-05-16T08:07:36,Punjab,Active,null
Lahore,Pakistan,2,1991-05-26,daniel.doyle2@email.com,Daniel,Male,Doyle,217,Silver,INSERT,2024-10-16T10:08:10Z,+923271310554,88907,2024-03-23T08:28:15,Punjab,Active,null
Hyderabad,Pakistan,3,1977-11-22,stephanie.miller3@email.com,Stephanie,Female,Miller,3462,Bronze,INSERT,2024-04-16T01:16:59Z,+923407381601,30926,2024-07-23T00:14:11,Sindh,Active,null
Peshawar,Pakistan,4,1982-04-26,allen.robinson4@email.com,Allen,Male,Robinson,2817,Silver,INSERT,2026-03-11T09:02:22Z,+923051924210,57052,2023-07-09T13:49:57,KPK,Active,null
Multan,Pakistan,5,1989-12-23,alyssa.gonzalez5@email.com,Alyssa,Female,Gonzalez,2962,Silver,INSERT,2024-05-11T15:33:23Z,+923157398888,92397,2023-06-11T20:05:57,Punjab,Active,null
Hyderabad,Pakistan,6,1986-10-06,joshua.robinson6@email.com,Joshua,Male,Robinson,3113,Bronze,INSERT,2025-08-02T03:36:25Z,+923465189878,23238,2023-06-13T08:28:32,Sindh,Active,null
Mardan,Pakistan,7,1998-03-29,robert.smith7@email.com,Robert,Male,Smith,584,Silver,INSERT,2024-06-18T15:57:01Z,+923376786911,99593,2024-03-05T09:43:13,KPK,Active,null
Hyderabad,Pakistan,8,1963-10-09,michael.peterson8@email.com,Michael,Male,Peterson,4562,Bronze,INSERT,2025-04-04T01:22:45Z,+923496815604,93886,2025-02-16T09:49:42,Sindh,Active,null
Karachi,Pakistan,9,1960-06-12,tyler.rogers9@email.com,Tyler,Male,Rogers,1728,Gold,INSERT,2025-01-03T03:19:41Z,+923143742291,18675,2024-10-08T14:36:21,Sindh,Inactive,null
Hyderabad,Pakistan,10,1987-07-13,anne.abbott10@email.com,Anne,Female,Abbott,2169,Bronze,INSERT,2026-01-28T15:41:46Z,+923246344213,28726,2025-03-21T23:24:19,Sindh,Active,null
